# 1 Tools概述

Tools 用于扩展大语言模型（LLM）的能力，使其能够与外部系统、API 或自定义函数交互，从而完成仅靠文本生成无法实现的任务（如搜索、计算、数据库查询等）。

特点：

1、增强 LLM 的功能：让 LLM 突破纯文本生成的限制，执行实际操作（如调用搜索引擎、查询数据库、运行代码等）

2、支持智能决策：在Agent 工作流中，LLM 根据用户输入动态选择最合适的 Tool 完成任务。

3、模块化设计：每个 Tool 专注一个功能，便于复用和组合（例如：搜索工具 + 计算工具 + 天气查询工具）

Tool 通常包含如下几个要素：

name ：工具的名称

description ：工具的功能描述

该工具输入的JSON模式

要调用的函数

return_direct ：是否应将工具结果直接返回给用户（仅对Agent相关）

步骤1：将name、description 和 JSON模式作为上下文提供给LLM

步骤2：LLM会根据提示词推断出需要调用哪些工具，并提供具体的调用参数信息

步骤3：用户需要根据返回的工具调用信息，自行触发相关工具的回调

# 2 自定义工具

## 2.1 两种自定义方式

第1种：使用@tool装饰器（自定义工具的最简单方式）

    装饰器默认使用函数名称作为工具名称，但可以通过参数name_or_callable 来覆盖此设置。

    同时，装饰器将使用函数的文档字符串作为工具的描述，因此函数必须提供文档字符串。

第2种：使用StructuredTool.from_function类方法

    这类似于@tool 装饰器，但允许更多配置和同步/异步实现的规范。

## 2.2 几个常用属性

name 
str 
必选的，在提供给LLM或Agent的工具集中必须是唯一的。

description 
str
可选但建议，描述工具的功能。LLM或Agent将使用此描述作为上下文，使用它确定工具的使用

args_schema
Pydantic
BaseModel
可选但建议，可用于提供更多信息（例如，few-shot示例）或验证预期参数。

return_direct 
boolean
仅对Agent相关。当为True时，在调用给定工具后，Agent将停止并将结果直接返回给用户。

## 2.3 具体实现

方式1：@tool 装饰器

举例1：

In [1]:
from langchain.tools import tool

@tool
def add_number(a:int,b:int)->int:
    """两个整数相加"""
    return a + b


print(f"name = {add_number.name}") #默认函数名称
print(f"args = {add_number.args}")  #参数
print(f"description = {add_number.description}") #默认函数描述
print(f"return_direct = {add_number.return_direct}") #仅对Agent相关。当为True时，在调用给定工具后，Agent将停止并将结果直接返回给用户。

res = add_number.invoke({"a":10,"b":20})
print(res)

name = add_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = 两个整数相加
return_direct = False
30


说明： return_direct参数的默认值是False。当return_direct=False时，工具执行结果会返回给Agent，让Agent决定下一步操作；而return_direct=True则会中断这个循环，直接结束流程，返回结果给用户。

举例2：通过@tool的参数设置进行重置

In [ ]:
from langchain.tools import tool

@tool(name_or_callable="add_two_number",description="two number add",return_direct=True)
def add_number(a:int,b:int)->int:
    """两个整数相加"""
    return a + b


print(f"name = {add_number.name}") #add_two_number
print(f"description = {add_number.description}") #two number add
print(f"args = {add_number.args}") #{'a': {'type': 'integer', 'description': 'a'}, 'b': {'type': 'integer', 'description': 'b'}}
print(f"return_direct = {add_number.return_direct}") #True

res = add_number.invoke({"a":10,"b":20})
print(res)

name = add_two_number
description = two number add
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
return_direct = True
30


补充：还可以修改args参数的说明

In [1]:
from langchain.tools import tool
from pydantic import BaseModel, Field

class FieldInfo(BaseModel):
    a :int = Field(description="第1个参数")
    b :int = Field(description="第2个参数")

@tool(name_or_callable="add_two_number",description="two number add",args_schema=FieldInfo,return_direct=True)
def add_number(a:int,b:int)->int:
    """两个整数相加"""
    return a + b


print(f"name = {add_number.name}")
print(f"description = {add_number.description}")
print(f"args = {add_number.args}")
print(f"return_direct = {add_number.return_direct}")


res = add_number.invoke({"a":10,"b":20})
print(res)

name = add_two_number
description = two number add
args = {'a': {'description': '第1个参数', 'title': 'A', 'type': 'integer'}, 'b': {'description': '第2个参数', 'title': 'B', 'type': 'integer'}}
return_direct = True
30


方式2：StructuredTool的from_function()

StructuredTool.from_function 类方法提供了比@tool 装饰器更多的可配置性，而无需太多额外的代码。

举例1：

In [4]:
from langchain_core.tools import StructuredTool
# 定义一个普通函数
def search_function(query: str):
    return "最后查询的结果"  

# 通过from_function方法创建StructuredTool实例
search1 = StructuredTool.from_function(
    func=search_function,
    name="Search",
    description="查询谷歌搜索引擎，并返回结果"
)

print(f"name = {search1.name}")
print(f"description = {search1.description}")
print(f"args = {search1.args}")

search1.invoke({"query": "中美AI发展现状"})

name = Search
description = 查询谷歌搜索引擎，并返回结果
args = {'query': {'title': 'Query', 'type': 'string'}}


'最后查询的结果'

举例2：更丰富的参数情况

In [6]:
from langchain_core.tools import StructuredTool
from pydantic import Field,BaseModel

class FieldInfo(BaseModel):
    query: str = Field(description="要检索的关键词")

def search_function(query: str):
    return "最后查询的结果"


search2 = StructuredTool.from_function(
    func=search_function,
    name="Search",
    description="查询谷歌搜索引擎，并返回结果",
    args_schema=FieldInfo,
    return_direct=True,
)

print(f"name = {search2.name}")
print(f"description = {search2.description}")
print(f"args = {search2.args}")
print(f"return_direct = {search2.return_direct}")

search2.invoke({"query": "中美AI发展现状"})

name = Search
description = 查询谷歌搜索引擎，并返回结果
args = {'query': {'description': '要检索的关键词', 'title': 'Query', 'type': 'string'}}
return_direct = True


'最后查询的结果'

## 2.4 工具调用举例

我们通过大模型分析用户需求，判断是否需要调用指定工具。

举例1：大模型分析调用工具

In [9]:
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 2.定义LLM模型
chat_model =ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 3.定义工具
tools = [MoveFileTool()]

# 4.这里需要将工具转换为openai函数，后续再将函数传入模型调用,invoke方法中会自动调用工具函数
functions = [convert_to_openai_function(t) for t in tools]

# print(functions[0])

# 5. 提供大模型调用的消息列表
messages = [HumanMessage(content="将文件a.txt移动到桌面")]

# 6.模型使用函数
response = chat_model.invoke(
    input = messages,
    functions=functions
)

print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a.txt","destination_path":"/Users/YourUsername/Desktop/a.txt"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 77, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D8zff5i4YYVHlvrbhSZe8tIO6QyLY', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019c59f9-ae56-7ff2-a8ce-7afb0aa76097-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 29, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


模型绑定工具，调用模型，传入Message对象。

作为对照，修改代码：

In [12]:
response = chat_model.invoke(
    [HumanMessage(content="今天的天气怎么样？")],
    functions=functions
)
print(response)

content='抱歉，我无法提供实时天气信息。你可以通过天气预报网站或应用程序查看今天的天气情况。需要其他帮助吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 74, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D8kFdbZZGQ01ff23Nkzbv3MUfMSMn', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c5671-3452-72e1-9b5c-9d02c67d08dd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 74, 'output_tokens': 32, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


调用工具说明

两种情况：

情况1：大模型决定调用工具

如果模型认为需要调用工具（如 MoveFileTool ），返回的 message 会包含：

content : 通常为空（因为模型选择调用工具，而非生成自然语言回复）。

additional_kwargs : 包含function_call字段，工具调用的详细信息.

例如：additional_kwargs={'function_call': {'arguments': '{"source_path":"a.txt","destination_path":"/Users/YourUsername/Desktop/a.txt"}', 'name': 'move_file'}, 'refusal': None} 

In [ ]:
AIMessage(
    content='', # 无自然语言回复
    additional_kwargs={
        'function_call': {
            'name': 'move_file', # 工具名称
            'arguments':
            '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}' # 工具参数
        }
    }
)

情况2：大模型不调用工具

如果模型认为无需调用工具（例如用户输入与工具无关），返回的 message 会是普通文本回复：

In [ ]:
AIMessage(
    content='我没有找到需要移动的文件。', # 自然语言回复
    additional_kwargs={'refusal': None} # 无工具调用
)

上面工具都还未调用

说明：

1、大模型和Agent的核心区别：是否涉及到工具的调用

2、针对于大模型仅仅能分析出要调用的工具，但是此工具（或函数）不能真正执行。

针对Agent：除了能分析要调用的工具之外，还可以执行工具（或函数）


举例2：确定工具并调用

In [14]:
# 定义LLM模型
chat_model =ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 定义工具
tools = [MoveFileTool()]

# 将工具转换为openai函数
functions = [convert_to_openai_function(t) for t in tools]

# 提供消息列表
messages = [HumanMessage(content="将本目录下的a.txt文件移动到C:/Users/huan.zheng/Desktop")]

# 模型调用
response = chat_model.invoke(
    input=messages,
    functions=functions
)
print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a.txt","destination_path":"C:/Users/huan.zheng/Desktop/a.txt"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 88, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D8zrDKfKNosUjyPe8j3eamNoCd4wq', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019c5a04-9e3b-72f0-a8be-d30579df68ce-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 88, 'output_tokens': 32, 'total_tokens': 120, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


分析下要调用哪个工具或函数

In [15]:
if "function_call" in response.additional_kwargs:
    function_name = response.additional_kwargs["function_call"]["name"]
    arguments = response.additional_kwargs["function_call"]["arguments"]
    print(f"调用了工具：{function_name}，\n 参数为：{arguments}")

else:
    print(f"模型回复：{response.content}")

调用了工具：move_file，
 参数为：{"source_path":"a.txt","destination_path":"C:/Users/huan.zheng/Desktop/a.txt"}


检查是否需要调用工具,tool_args需要转jason才能给tool.run

In [17]:
import json

if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具: {tool_name}, \n参数: {tool_args}")
else:
    print(f"模型回复: {response.content}")

调用工具: move_file, 
参数: {'source_path': 'a.txt', 'destination_path': 'C:/Users/huan.zheng/Desktop/a.txt'}


(2) 实际执行工具调用

In [19]:
from langchain_classic.tools import MoveFileTool

if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args) # 执行工具
    print("工具执行结果:", result)

工具执行结果: File moved successfully from a.txt to C:/Users/huan.zheng/Desktop/a.txt.


以上是代码tool.run调用工具.如果是Agent则不需要，agent可自行调用